# RAY-IMAGE v0.1 — 12-Class Learning Test

This notebook trains the tiny RAY-IMAGE prototype on a richer synthetic dataset, then generates and numerically evaluates all 12 color/shape classes.

Select **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. Select a GPU runtime and reconnect.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
%cd /content
!rm -rf anime-ai-companion
!git clone https://github.com/Rishidev-20thcenturey/anime-ai-companion.git
%cd /content/anime-ai-companion
!pip install -q -r requirements.txt

In [ ]:
!python -m ray_image.train_smoke

In [ ]:
!rm -rf data/toy
!python tools/make_toy_dataset.py --output data/toy --samples 2048 --size 64 --seed 1337

In [ ]:
!python -m ray_image.train_vae --manifest data/toy/manifest.jsonl --steps 1200 --batch-size 32 --save /content/ray_vae_v0_2.pt

In [ ]:
!python -m ray_image.train_generator --manifest data/toy/manifest.jsonl --vae /content/ray_vae_v0_2.pt --steps 5000 --batch-size 16 --save /content/ray_image_v0_2_trained.pt

In [ ]:
from pathlib import Path
prompts = [
    ('red_circle', 'a red circle'), ('red_square', 'a red square'), ('red_triangle', 'a red triangle'),
    ('green_circle', 'a green circle'), ('green_square', 'a green square'), ('green_triangle', 'a green triangle'),
    ('blue_circle', 'a blue circle'), ('blue_square', 'a blue square'), ('blue_triangle', 'a blue triangle'),
    ('yellow_circle', 'a yellow circle'), ('yellow_square', 'a yellow square'), ('yellow_triangle', 'a yellow triangle'),
]
for name, prompt in prompts:
    out = Path('/content/ray_suite') / f'{name}.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    !python -m ray_image.generate --checkpoint /content/ray_image_v0_2_trained.pt --prompt "{prompt}" --steps 50 --seed 42 --output "{out}"
    print(name, 'exists=', out.exists(), 'bytes=', out.stat().st_size if out.exists() else '-')

In [ ]:
!python tools/evaluate_toy_suite.py --dir /content/ray_suite

## Target

A useful toy result is that the predicted class changes with the prompt rather than collapsing to one class. The suite score is only a coarse pixel-level diagnostic; success here means the architecture demonstrably learns a small text-to-image task before we move to real images.